# Lesson 6 — Kings Pawn Opening 1...e5 (as Black)
**Project 1500 | Priority #6**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE, flipped=True):
    svg = chess.svg.board(board, arrows=arrows or [], size=size, flipped=flipped)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as black (April 2025+)

In [ ]:
def load_rapid_games_as_black(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            black = g.get('black', {})
            if black.get('username', '').lower() != username.lower(): continue
            result = black.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_black_games = load_rapid_games_as_black(USERNAME)
print(f'Loaded {len(all_black_games)} rapid games as black')

## Step 2 — Detect games where white played early Qh5 and track black's response

We detect this pattern by position: white queen on h5 within the first 6 half-moves.

In [ ]:
qh5_nc6_games = []  # black played Nc6 (correct)
qh5_g6_games  = []  # black played g6 (blunder)
qh5_other     = []  # other responses

for g in all_black_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        qh5_seen = False
        for i, move in enumerate(moves[:8]):
            if i % 2 == 0:  # white's move
                if board.san(move) == 'Qh5':
                    qh5_seen = True
            elif qh5_seen:  # black's response to Qh5
                resp = board.san(move)
                if resp == 'Nc6':
                    qh5_nc6_games.append(g)
                elif resp == 'g6':
                    qh5_g6_games.append(g)
                else:
                    qh5_other.append(g)
                break
            board.push(move)
    except Exception:
        pass

total_qh5 = len(qh5_nc6_games) + len(qh5_g6_games) + len(qh5_other)
print(f'Games where white played Qh5 early: {total_qh5}')

for label, games_list in [('Nc6 (correct)', qh5_nc6_games), ('g6 (blunder)', qh5_g6_games), ('other', qh5_other)]:
    c = Counter(g['outcome'] for g in games_list)
    t = len(games_list)
    wr = 100 * c['win'] / t if t else 0
    print(f'  {label:18s}: {t:3d} games  Win%: {wr:.1f}%')

if total_qh5 < 10:
    print('\nNote: small sample — treat as directional.')

## Step 3 — Board diagrams *(illustrative example lines)*

In [ ]:
# [ILLUSTRATIVE] After 1.e4 e5 2.Qh5 — the Scholar's Mate setup
# Verified legal: e4 e5 Qh5
AFTER_QH5 = ['e4','e5','Qh5']

show_board(
    board_after(AFTER_QH5),
    arrows=[
        chess.svg.Arrow(chess.B8, chess.C6, color='#27ae60'),  # Nc6 — correct
        chess.svg.Arrow(chess.G7, chess.G6, color='#c0392b'),  # g6 — blunder
    ],
    caption='[ILLUSTRATIVE] After 2.Qh5 — Green=Nc6 (defends e5, develops) | Red=g6 (allows Qxe5+ fork)'
)

In [ ]:
# [ILLUSTRATIVE] Why g6 loses immediately — Qxe5+ forks king and rook
# Verified legal: e4 e5 Qh5 g6 Qxe5+ (queen on h5 slides along rank 5 to e5)
G6_BLUNDER = ['e4','e5','Qh5','g6','Qxe5+']

show_board(
    board_after(G6_BLUNDER),
    arrows=[
        chess.svg.Arrow(chess.E5, chess.H8, color='#c0392b'),  # queen forks king and h8 rook
    ],
    caption='[ILLUSTRATIVE] After g6?? Qxe5+ — queen forks king and h8 rook. Material loss is immediate.'
)

In [ ]:
# [ILLUSTRATIVE] The correct response — Nc6 defends e5
# Verified legal: e4 e5 Qh5 Nc6
NC6_CORRECT = ['e4','e5','Qh5','Nc6']

show_board(
    board_after(NC6_CORRECT),
    arrows=[
        chess.svg.Arrow(chess.C6, chess.E5, color='#27ae60'),  # Nc6 defends e5
        chess.svg.Arrow(chess.H5, chess.H5, color='#c0392b'),  # white queen is misplaced
    ],
    caption='[ILLUSTRATIVE] After Nc6 — e5 is defended, white queen on h5 is a liability. Black is fine.'
)

## Step 4 — Summary

In [ ]:
nc6_wr = 100 * Counter(g['outcome'] for g in qh5_nc6_games)['win'] / max(len(qh5_nc6_games), 1)
g6_wr  = 100 * Counter(g['outcome'] for g in qh5_g6_games)['win']  / max(len(qh5_g6_games),  1)

print('KINGS PAWN / QH5 TRAP — SUMMARY')
print('=' * 50)
print(f'\nGames with early Qh5: {total_qh5}')
print(f'  Nc6 (correct): {len(qh5_nc6_games)} games, {nc6_wr:.1f}% win rate')
print(f'  g6  (blunder): {len(qh5_g6_games)} games, {g6_wr:.1f}% win rate')
print()
print('RULES:')
print('  1. When white plays Qh5 — ALWAYS respond with Nc6')
print('  2. NEVER play g6 — allows Qxe5+ forking king and h8 rook')
print('  3. After Nc6, white\'s queen on h5 is a liability — develop normally')
print('  4. Same rule applies to 1.e3 e5 2.Qh5 — identical trap, identical fix')